# 13 — Comparación de métodos entre tareas

Reúne, para cada tarea implementada, los cuatro métodos tabulares con el mismo protocolo dentro de cada tarea:
mismos estados, acciones, presupuesto, semillas (0–4) e inicios de evaluación; solo cambia la regla de
actualización. Métrica común: **media de las últimas 5 evaluaciones greedy** (robusta a la oscilación de la
conducción), con IC 95 % bootstrap sobre semillas. Una diferencia se declara solo si los IC no se solapan.

| tarea | éxito | presupuesto | α | fuente |
|---|---|---|---|---|
| Persecución (kit, [5, 40] m) | captura en < 40 pasos | 80 000 | 0.1 para todos (sin barrido) | `runs_kit_final_algorithms`, `runs_kit_final` (notebook 07) |
| Tiro a puerta (con arquero) | gol | 50 000 | barrido con ε decreciente | `runs_shooting` (notebook 10) |
| Conducción (36 estados) | avance de 30 m | 60 000 | barrido por esquema | `runs_dribbling` (notebook 11) |
| Cooperación 2v1 (324 estados) | posesión > 50 pasos y ≥ 3 pases | 60 000 | barrido por esquema | `runs_passing` (notebook 12) |

Las explicaciones de cada diferencia están en los
notebooks de cada tarea, junto al diagnóstico que las respalda.

In [1]:
import os, json, glob
import numpy as np
from IPython.display import display, Markdown
os.chdir("/workspace") if os.path.isdir("/workspace/src") else None
from src.task_analysis import bootstrap_ci

ART = "notebooks/artifacts"
METHODS = ["qlearning", "sarsa", "mc_first_visit", "mc_first_visit_alpha"]
METHOD_ES = {"qlearning": "Q-Learning", "sarsa": "SARSA", "mc_first_visit": "MC (promedio)",
             "mc_first_visit_alpha": "MC (α constante)"}

def stats(values):
    v = np.asarray(values, float)
    return {"mean": float(v.mean()), "ci95": bootstrap_ci(v), "per_seed": v.tolist()}

def pursuit(cond_dir):
    last5, auc = [], []
    for f in sorted(glob.glob(f"{cond_dir}/seed_*/eval.json")):
        ev = [e["capture_rate_lt40"] for e in json.load(open(f))["evals"]]
        last5.append(np.mean(ev[-5:])); auc.append(np.mean(ev))
    return {"last5_mean": stats(last5), "auc": stats(auc)}

def task(summary, name, variant):
    v = summary[name]["variants"][variant]
    return {"last5_mean": stats(v["last5_mean"]["per_seed"]), "auc": stats(v["auc"]["per_seed"])}

shoot = json.load(open(f"{ART}/shooting_summary.json"))["conditions"]
drib = json.load(open(f"{ART}/dribbling_summary.json"))["conditions"]
pas = json.load(open(f"{ART}/passing_summary.json"))["conditions"]
table = {"decay": {}, "constant": {}}
table["decay"]["Persecución"] = {m: pursuit(f"{ART}/runs_kit_final_algorithms/kit_algo_{m}") for m in METHODS}
table["constant"]["Persecución"] = {"qlearning": pursuit(f"{ART}/runs_kit_final/kit_qlearning_eps_const_0.1")}
for sched, tag in (("decay", "eps_decay_1_to_0.1"), ("constant", "eps_const_0.1")):
    table[sched]["Tiro a puerta"] = {m: task(shoot, f"shooting_{m}_{tag}", "keeper") for m in METHODS}
    table[sched]["Conducción"] = {m: task(drib, f"dribbling_{m}_{tag}", "default") for m in METHODS}
    table[sched]["Cooperación 2v1"] = {m: task(pas, f"passing_{m}_{tag}", "default") for m in METHODS}
json.dump({"metric": "mean of the last 5 greedy evaluations (success); auc = mean over all evaluations",
           "ci": "95% percentile bootstrap over the 5 training seeds", "table": table},
          open(f"{ART}/comparison_summary.json", "w"), indent=2, ensure_ascii=False)

## Tabla principal: $\varepsilon$ decreciente

En negrita, el mejor método de cada tarea y los que no se distinguen de él (IC solapados).

In [2]:
def render(sched, metric="last5_mean"):
    rows = ["| tarea | " + " | ".join(METHOD_ES[m] for m in METHODS) + " |", "|---|---|---|---|---|"]
    for t, cells in table[sched].items():
        best = max(cells.values(), key=lambda c: c[metric]["mean"])[metric]
        out = []
        for m in METHODS:
            if m not in cells:
                out.append("—"); continue
            s = cells[m][metric]
            txt = f"{100 * s['mean']:.1f} [{100 * s['ci95'][0]:.1f}, {100 * s['ci95'][1]:.1f}]"
            tie = not (s["ci95"][1] < best["ci95"][0] or best["ci95"][1] < s["ci95"][0])
            out.append(f"**{txt}**" if tie else txt)
        rows.append(f"| {t} | " + " | ".join(out) + " |")
    return "\n".join(rows)
display(Markdown("**Éxito, media de las últimas 5 evaluaciones (%):**\n\n" + render("decay")))
display(Markdown("**AUC: éxito medio a lo largo del entrenamiento (%), mayor = aprende antes:**\n\n" + render("decay", "auc")))

**Éxito, media de las últimas 5 evaluaciones (%):**

| tarea | Q-Learning | SARSA | MC (promedio) | MC (α constante) |
|---|---|---|---|---|
| Persecución | **84.7 [84.5, 85.1]** | **85.3 [84.8, 85.8]** | **85.3 [84.4, 86.1]** | 45.8 [41.6, 50.1] |
| Tiro a puerta | **92.2 [91.7, 92.6]** | **92.2 [91.7, 92.7]** | **92.8 [92.4, 93.4]** | **91.8 [90.6, 92.5]** |
| Conducción | **97.7 [96.3, 99.0]** | **94.3 [89.8, 97.7]** | **90.4 [79.0, 98.9]** | 30.1 [16.3, 45.7] |
| Cooperación 2v1 | **64.8 [61.3, 68.1]** | 54.2 [51.0, 56.1] | 7.8 [2.9, 14.9] | 20.7 [9.8, 31.5] |

**AUC: éxito medio a lo largo del entrenamiento (%), mayor = aprende antes:**

| tarea | Q-Learning | SARSA | MC (promedio) | MC (α constante) |
|---|---|---|---|---|
| Persecución | 77.0 [76.6, 77.4] | **83.3 [82.8, 83.9]** | **84.3 [83.1, 85.3]** | 43.2 [41.9, 44.6] |
| Tiro a puerta | 90.9 [90.2, 91.6] | 90.5 [89.8, 91.1] | **92.6 [92.1, 93.0]** | 90.0 [89.3, 90.7] |
| Conducción | **92.4 [90.5, 94.3]** | 76.2 [73.5, 79.9] | 70.8 [57.6, 80.5] | 22.3 [18.9, 26.6] |
| Cooperación 2v1 | **44.9 [42.5, 46.5]** | 28.2 [25.3, 30.2] | 2.6 [1.5, 3.6] | 9.2 [4.8, 14.2] |

## $\varepsilon$ constante 0.1

En la persecución solo Q-Learning se entrenó con $\varepsilon$ constante (la ablación del notebook 07).

In [3]:
display(Markdown(render("constant")))

| tarea | Q-Learning | SARSA | MC (promedio) | MC (α constante) |
|---|---|---|---|---|
| Persecución | **84.9 [84.4, 85.4]** | — | — | — |
| Tiro a puerta | 86.6 [85.1, 88.1] | 86.5 [85.0, 88.0] | **93.2 [92.6, 93.7]** | 86.0 [84.3, 87.5] |
| Conducción | 89.7 [83.8, 95.5] | **96.8 [95.8, 97.8]** | 70.6 [52.0, 87.6] | 18.8 [7.1, 33.8] |
| Cooperación 2v1 | **65.5 [64.2, 67.4]** | 54.2 [51.9, 56.3] | 26.1 [18.0, 39.9] | 16.7 [6.8, 27.6] |

## Lectura

- **Persecución** (determinista, recompensa densa, sin riesgo): al final, Q-Learning, SARSA y MC con promedios no
  se distinguen; el techo lo pone la representación (óptimo exacto 87.9 %, notebook 07), no la regla de
  actualización. SARSA y MC con promedios aprenden antes que Q-Learning (AUC).
- **Tiro a puerta** (casi una sola decisión, resultado estocástico): con ε decreciente los cuatro terminan igual;
  MC con promedios aprende antes (AUC) y es el único que no se retrasa con ε constante. Q-Learning y SARSA son
  idénticos por construcción (los tiros son terminales). Diagnóstico: notebook 10, §7.
- **Conducción** (determinista, episodios largos, estados agregados): con ε decreciente, Q-Learning, SARSA y MC
  con promedios no se distinguen al final (el IC de MC es muy ancho, 79–99 %), pero Q-Learning aprende mucho antes
  (AUC). Con ε constante, SARSA es el mejor. Con 36 estados, MC con promedios congela su política en algunas
  semillas; con 245 estados llega a 97.8 %, y Q-Learning y SARSA a 100 %. Diagnóstico: notebook 11, §8.
- **Cooperación 2v1** (estocástica, 60 pasos, riesgo de quite): Q-Learning es el mejor con ambas exploraciones y
  aprende antes; SARSA queda unos 11 puntos atrás y los dos MC fallan (MC con promedios termina la mayoría de los
  episodios con un despeje). SARSA no fue más prudente que Q-Learning pese al riesgo real. Diagnóstico: notebook 12, §8.
- **MC** (ambas variantes) falla en la cooperación 2v1, y MC con α constante también en la persecución y la
  conducción: en las tres tareas de episodios largos. Solo funciona bien en el tiro a puerta, de un paso. La causa
  no está aislada.